# Graph Algorithms

A graph is just a way to represent relationships between things (like a map of cities connected by roads). 

A graph G has two elements:

- Vertices (aka nodes): the "things." We call the set $V$, and $n = |V|$ is the amount
- Edges: the "relationships" between things. We call the set $E$, and $m = |E|$ is the amount

The number of edges is bounded: $0 \leq m \leq O(n^2)$ because in the worst case every vertex is connected to every other vertex, and that's $\frac{n(n-1)}{2}$ edges, whichs is a complete graph.

## Directed vs. Undirected

An **undirected** graph means the relationship goes both ways: if there's a road from city A to city B, we can also drive from B to A. And edge is just written as $(v_i, v_j)$.

A **directed graph** (or diagraph) means edges have a specified direction, like a one-way street. They're drawn with arrows.

## Degree

The **degree** of a vertex is how many edges touch it. In a directed graph, this plits into two:

- In-degree: how many edges come in to the vertex
- Out-degree: how many edges go out from it

Also, a **sparse graph** is one without many edges ($m = O(n)$), while a **dense** graph has many edges, closer to $O(n^2)$.

## How do we store a graph in code?

### Option 1: Adjacency matrix

We make an $n \times n$ 2D array $M$, where $M[i][j] = 1$ if there's an edge between $v_i$ and $v_j$, and 0 if there isn't.

```
Graph:  1——2——5
        |  |
        3——4

Matrix:   1  2  3  4  5
       1[ 0  1  1  0  0 ]
       2[ 1  0  1  1  1 ]
       3[ 1  1  0  1  0 ]
       4[ 0  1  1  0  0 ]
       5[ 0  1  0  0  0 ]
```

It's symmetric for undirected graphs. The space is $O(n^2)$, because we're always storing the full grid, even if most of it is zeros.

### Option 2: Adjacency Lists

Instead of a big grid, each vertex gets its own linked list of neighbors. We store an array ```adj[1...n]``` where ```ad[i]``` is the head of the list for vertex $v_i$

```
Graph:  1——2——5
        |  |
        3——4

adj[1]: 2 -> 3
adj[2]: 1 -> 4 -> 5 -> 3
adj[3]: 1 -> 2 -> 4
adj[4]: 2 -> 3
adj[5]: 2
```

The space cost is $O(n + m)$, because we store each vertex once, and each edge once (or twice for undirected).

## Which storage option do we use?

There are two main operations that we will constantly be using when dealing with graphs:

1. Is there an edge between $v_i$ and $v_j$?
- Matrix: $O(1)$, we just look up $M[i][j]$
- List: Linear in the size of $v_i$'s list


2. List all the neighbors of $v_i$
- Matrix: $O(n)$, we scan the whole $i$-th row looking for 1s
- List: linear in the size of $v_i$'s list

So: adjacency lists win for sparse graphs, and they're what almos tevery graph algorithm uses. The matrix is only really worth it if the graph is dense and we need constant-time lookups.

## BFS

Given a graph $G$ and a source vertex $s$, finnd the shortest path from $s$ to every other vertex. This is called the **Single-Source Shortest Paths**. For now, shortest means fewest edges.

Imagine dropping a stone in a pond, then the ripples spread out in rings. First one step way, then two steps away, then three, then... you get the idea. That's basically **Breadth-First Search**.

We start at $s$, visit all its neighbors (distance 1), the visit all their unvisited neighbors (distance 2), and so on.

BFS always visits all vertices of distance $k$ from $s$ before visiting any vertex of distance $k + 1$.

To implement it, we need a queue to keep track of who to visit next. Vertices will get added to the back and then processed from the front.

For each vertex, we track:
- ```color[v]```: Where unvisited = white, visited but not explored = blue, completed = red
- ```d[v]```: shortest distance from $s$ to $v$
- ```pre[v]```: predecessor of $v$ on the shortest path


```
BFS(G, s):
    for each vertex v:
        color[v] = white
        d[v] = infinity
        pre[v] = null
    
    d[s] = 0
    color[s] = blue
    enqueue(Q, s)

    while Q is not empty:
        u = dequeue(Q)
        for each v in adj[u]:
            if color[v] == white:
                color[v] = blue
                d[v] = d[u] + 1
                pre[v] = u
                enqueue(Q, v)
        color[u] = red
```

The time complexity is $O(n + m)$.

This is how it works. Let's say this is our graph:
```
s — a — c
|       |
b — d — t
```

BFS would do:
```
Start:       Q = [s],          d[s]=0
Round 1:     dequeue s         d[a]=1, d[b]=1,   Q = [a, b]
Round 2:     dequeue a         d[c]=2,            Q = [b, c]
             dequeue b         d[d]=2,            Q = [c, d]
Round 3:     dequeue c         d[t]=3,            Q = [d, t]
             dequeue d         (t already visited)
```

The ```pre``` array at the end gives us a shortest path tree, which is a tree rooted at $s$ where follwing $pre$ pointers back from any vertex gives us the shortest path to $s$.

```
        s
       / \
      a   b
      |   |
      c   d
      |
      t
```

## Directed Acyclic Graph

A **DAG** is a directed graph with no cycles, so no matter where we start, we can never follow the edges and end up back where we started.

For example, course prerequisities: CS 1400 -> CS 1410 -> CS 2420, and CS 2100 -> CS 4150

### Topological order

Because a DAG has no cycles, we can always arrange all its vertices in a line such that every edge points left to right. This is called toplogical order.

For example:

```
Graph:      a
           / \
          c   b
           \ / \
            d   e
           / \
          f   h

One valid topological order:  a, b, c, e, d, f, h
Another valid one:            a, c, b, e, d, f, h
```

The actual definition is: for every directed edge $(u, v)$, $u$ always appear before $v$ in ordering.

The implement this algorith, we repeatedly find a vertex with in-degree zero (so that nothing points to it), output it, remove it and all its outgoing edges from the graph, then repeat.

So we:
- Maintain an ```indegree``` count for every vertex
- Use a queue $Q$ to store all current in-degree-zero vertices


```
TopologicalSort(G):
    for each vertex v:
        indegree[v] = 0

    for each vertex u:
        for each vertex v in adj[u]:
            indegree[v]++
    
    Q = not empty
    for each vertex v:
        if indegree[v] == 0:
            enqueue(Q, v)

    while Q is not empty:
        u = dequeue(Q)
        output u
        for each v in adj[u]:
            indegree[v]--
            if indegree[v] == 0:
                enqueue(Q, v)
```

The time is $O(n + m)$.

This is an example, for the graph:

```
Graph:  a → c → d
        b → d → e
            b → e
```

```
Q = [a, b]

dequeue a → output a, remove a→c: indegree[c] becomes 0 → enqueue c
Q = [b, c]

dequeue b → output b, remove b→d and b→e: indegree[d]=1, indegree[e]=0 → enqueue e
Q = [c, e]

dequeue c → output c, remove c→d: indegree[d]=0 → enqueue d
Q = [e, d]

dequeue e → output e
dequeue d → output d

Result: a, b, c, e, d 
```

## Shortest Paths in DAGs

Now, we can add weights to edges. Each edge $(u, v)$ has a cost $w(u, v)$ and the length of a path is the sum of all edge weights along it.

BFS works when every edge costs the same, but if edges have different weights, taking fewer edges is not necessarily better.

```
s —1— a —1— t        path length = 2
s ———5——— t           path length = 5
```

In this graph, BFS would say that the first one is two edges and the second is one edge, and would pick the second, but the first path is cheaper than the second one.

Because a DAG has no cycles, if we process vertices if **topological order**, we're guaranteed that by the time we process vertex $v$ we have already finalized the shortest distances to all of $v$'s predecessors. No vertex can give us a better path later, because there are no cycles.

### Dynamic Programming

We can define $d[v] =$ shortest path length from $s$ to $v$.

Base case: $d[s] = 0$

Dependency relation: to get to $v$, we must have come from some predecessor $u$ where $(u, v)$ is an edge. So:

$d[v] = min {d[u] + w(u, v)}, (u,v) \in G$

```
ShortestPathDAG(G, s):
    for each vertex v:
        d[v] = infinity
        pre[v] = null

    d[s] = 0
    topological sort

    for each vertex u in topological order:
        for each v in adj[u]:
            if d[u] + w(u, v) < d[v]:
                d[v] = d[u] + w(u,v)
                pre[v] = u
```

The time is $O(n + m)$.

An example:

```
3       1       1       2       2
s ——→ a ——→ b ——→ c ——→ d ——→ t
 \                 ↑               
  \_______2________/
```

```
d[s] = 0,  d[a] = d[b] = d[c] = d[d] = d[t] = ∞

Process s:  edge s→a: d[a] = 0+3 = 3
            edge s→c: d[c] = 0+2 = 2 

Process a:  edge a→b: d[b] = 3+1 = 4

Process b:  edge b→c: d[c] = min(2, 4+1) = 2

Process c:  edge c→d: d[d] = 2+2 = 4
            
Process d:  edge d→t: d[t] = 4+2 = 6

Result: s→c→d→t with total cost 6
```

If we have negative weights, the algorithm works the same. If we want to find the longest path, we change min to max and initialize everythng to negative infinity.

## Dijkstra's Algorithm

Still used to find the shortest paths from source $s$ to all other vertices. But now the graph can have cycles, so we can't use topological order. The constraint is that all weights must be non-negative $w(u,v) \geq 0$.

In the DAG algorithm, topological order gave us the safe sequence to process vertices by processing $u$ before any vertex that depended on $u$.

Dijkstra's main idea is that, without topological order, we can use distance as our guide instead. The vertex with current smallest $d[v]$ is safe to finalize (because all edge weights are non-negative, no future path can ever make it cheaper.)

We use:

- $R$: vertices whose shortest path is finalized
- $Q$: everyone else, stored in a min-heap keyed by their current best $d[v]

In each iteration, we pull the vertex $u$ with minimum $d[u]$ out of $Q$, add it to $R$, and update its neighbors. This is called relaxing the edges out of $u$. For each neighbor, $v$ of $u$, we check if going through $u$ cheaper than our current best path to $v$.

```
if d[v] > d[u] + w(u,v):
    d[v] = d[u] + w(u,v)
    pre[v] = u
    decreaseKey(Q, v, d[v])
```

```decreaseKey``` is a heap operations that moves $v$ up in the min-heap since its key got smaller, and it costs $O(\log n)$.

The algorithm is:
```
Dijkstra(G, s):
    for each vertex v:
        d[v] = infinity
        pre[v] = null

    d[s] = 0
    Build min-heap Q with all vertices, keyed by d[v]

    while Q is not empty:
        u = deleteMin(Q)
        for each v in adj[u]:
            if d[v] > d[u] + w(u,v):
                d[v] = d[u] + w(u,v)
                pre[v] = u
                decreaseKey(Q, v, d[v])
```

The time is $O((n + m) \log n)$, because ```deleteMin``` happens $n$ times, and ```decreasKey``` happens at most $m$ times.

An example:

```
        10
   s ————————→ d
   |  \    ↗   |
   5   3  /    |
   ↓    ↘/ 9   | 2
   a     b ————→ c
     \  2↗  7↗
      ——→——→
```

```
Initial:  d[s]=0, d[a]=∞, d[b]=∞, d[c]=∞, d[d]=∞
Q = [s, a, b, c, d]

Pull s (d=0):   relax s-a: d[a]=5
                relax s-b: d[b]=3
                relax s-d: d[d]=10
Q = [b, a, d, c]

Pull b (d=3):   relax b-a: d[a]=min(5, 3+2)=5  no update
                relax b-d: d[d]=min(10,3+9)=10  no update
                relax b-c: d[c]=3+7=10
Q = [a, d, c]

Pull a (d=5):   relax a-c: d[c]=min(10,5+7)=10  no update
Q = [d, c]

Pull d (d=10):  relax d→c: d[c]=min(10,10+2)=10  no update... 
Q = [c]

Pull c (d=10): done.
Result: d[a]=5, d[b]=3, d[c]=10, d[d]=10
```

## Bellman-Ford

Same problem, but now edge weights can be negative. There's one assumption though: the graph has no negative cycles (in order words, no cycles whose total weight is negative).

If there are $n$ vertices and no cycles, any shortest path vists each vertex at most once, so it has at most $n-1$ edges.

We can't use topological order because the graph has cycles, and we can't use a heap greedily as negative weights break the guarantee. Bellman-Ford does a "brute-force"n approach. We run $n-1$ rounds, in each round, check every single edge and relax it.

After round $k$, $d[v]$ is guaranteed to be the shortest path to $v$ using at most $k$ edges, and by round $n-1$ we have covered all possible shortest paths.

```
Bellman-Ford(G, s):
    for each vertex v:
        d[v] = infinity
        pre[v] = null
    
    d[s] = 0
    for i = 1 to n-1:
        for each vertex u:
            for each vertex v in adj[u]:
                if d[v] > d[u] + w(u,v):
                    d[v] = d[u] + w(u,v)
                    pre[v] = u
```

The time is $O(n(n+m))$.

An example:

```
       6        5
   s ————→ d ————→ c
    \      ↑  -2  ↗
     7    -2  ↗  7
      \   ↗  ↗
       → a ——→ b
           9
```

```
Initial: d[s]=0, d[a]=∞, d[b]=∞, d[c]=∞, d[d]=∞

Round 1 (at most 1 edge):
  relax s-d: d[d] = 6
  relax s-a: d[a] = 7
  relax a-b: d[b] = 7+9 = 16  (if d[a] already updated this round)
  ...

Round 2 (at most 2 edges):
  relax d-a: d[a] = min(7, 6+(-2)) = 4
  relax a-c: d[c] = 4+(-2) = 2... 
  ...

Round 3 (at most 3 edges):
  relax a-c: d[c] gets updated again with the better d[a]
  ...
```

## Minimum Spanning Trees

Now we aren't looking for the shortest path from $s$ to everyone, but what's the cheapest way to connect all vertices together. Like, imagine you work at a telecom company and we need to lay cables to connect $n$ cities, every pair of cities has a cost to connect, ad we don't need a direct cable between every pair, but the network needs to be connected so any city can reach any other.

### Spanning Tree

A spanning tree $T$ of a graph $G$ ig a subgraph that:
- Is a tree (has no cycles)
- Contains every vertex of $G$
- Connects all vertices

A tree with $n$ vertices alwayas has exactly $n-1$ edges. A Minimum Spanning Tree is the spanning tree with the smallest total edge weight among all possible trees.

```
Graph:          MST:
  4   3           4   3
a———b———c       a———b———c
|   |   |               |
6  10  17               |
|   |   |           2   |
e———i———d       e   i———d
  13  2
```

### The Cut Property

A cut is any partition of vertices into two groups $S$ and $V \ S$. For any substet $S$ of vertices, the minimum weight edge that crosses the cut (one endpoint in $S$, one outside) must be in some MST.

If we've split the graph into two groups and we need to connect them, we should always use the cheapest bridge available.

```S          V\S
   o     o  21  o
   o     o  15  o
   o     o  6 - minimum! must be in MST
   o     o  10  o
```

### Prim's Algorithm

Prim's grows the MST one vertex at a time, starting from an arbitrary source $s$. At every step, we grab the cheapest edge that connects a vertes already in the tree to a vertex not yet in it.

For each vertex $v$ not yet in the tree, we maintain:
- key[v] - the minimum weight edge connecting $v$ to any vertex currently in the tree
- pre[v] - which tree vertex offers that minimum edge

We use a min-heap $Q$ keyed by $key[v]$ to always efficiently grab the cheapest next vertexx.

```
PrimMST(G):
    for each vertex v:
        flag[v] = 0
        key[v] = infinity
        pre[v] = null

    pick arbitrary s; key[s] = 0
    build min-heap Q for all vertices keyed by key[v]

    while Q is not empty:
        u = deleteMin(Q)
        flag[u] = 1
        for each v in adj[u]:
            if flag[v] == 0 and key[v] > w(u,v):
                key[v] = w(u,v)
                pre[v] = u
                decreaseKey(Q, v, key[v])
```

The time is $O((n+m) \log n). 

An example:

```
Graph:  s———4——b———8———c———7———d
         \     | \ 2 /   \     |
          8   11   i       \   9
            \  | / 6 \       \ |
               h———1———g———2———f
               7
```

```
Pull s:   neighbors b (w=4), h (w=8)
          key[b]=4, key[h]=8

Pull b:   neighbors c (w=8), i (w=11), h(w=8 no update since 8=8)
          key[c]=8, key[i]=11

Pull c:   neighbors d (w=7), i (w=2), f (w=4)
          key[d]=7, key[i]=min(11,2)=2, key[f]=4

Pull i:   neighbors h (w=7), g(w=6)
          key[h]=min(8,7)=7, key[g]=6

Pull f:   neighbors g (w=2), d(w=14)
          key[g]=min(6,2)=2

Pull g:   neighbors h (w=1)
          key[h]=min(7,1)=1

Pull h:   no updates needed

Pull d:   done

MST edges: s-b, b-c, c-i, c-f, f-g, g-h, c-d
```
